# Decoders 

$\renewcommand{\ket}[1]{|#1\rangle}$

In [ ]:
!pip install cudaq -q
!pip install torch scikit-learn galois ipywidgets -q


> **Note:** Run the cell below to import all required packages.
> If you installed packages above, restart the kernel first
> (**Runtime → Restart session** in Colab, or **Kernel → Restart** in Jupyter).

In [ ]:
import sys
import os
from itertools import product

import numpy as np
import matplotlib.pyplot as plt
import requests
import bz2
import galois
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.model_selection import train_test_split

import cudaq
from cudaq import spin
from cudaq.qis import *

## To install cudaq-qec (if not already installed), uncomment and run:
## !pip install cudaq-qec -q
import cudaq_qec as qec

from Images.decoder.decoder_widget import display_widget
from Images.decoder.bp import run_decoder, parse_csr_mat, parse_H_csr, parse_obs_csr

cudaq.set_target('nvidia')

## Decoding Decoded

Remember that a QEC round involves four main steps:
* Encoding logical qubits and sending the system through a noisy channel, which is often just a duration of time where the qubits are exposed to potential sources of error
* Measuring syndrome data
* Decoding the syndrome to identify where an error occurred and what instructions to send to the QPU to fix the error
* Correcting the error

<img src="https://github.com/osbama/KBM608/blob/main/hands-on/hands-on-7-images/decoding.png?raw=1" alt="Diagram showing the four main steps of a QEC round: encoding, syndrome measurement, decoding, and error correction, with the decoder receiving syndrome data and sending corrections to the QPU" style="width: 900px;"/>

The decoding step is very challenging and is considered one of the primary limitations for QEC. This is because decoding requires measurements on a QPU, data transfer to the supercomputer, decoding on the supercomputer, and then data transfer back to the QPU.  The time available for this is called the **decoding window** and varies based on a number of factors such as the qubit modality, data transfer rates, and the volume of information that needs to be decoded.

Directly competing with speed is accuracy. If a decoder is inaccurate, errors will be missed or introduced each QEC round and will propagate to ruin the computation. High-distance codes are necessary for accuracy, but unfortunately introduce high-qubit overheads and make decoding much more challenging. Advances in QEC code design and low-latency integration between AI supercomputers and QPUs alleviate pressure on the decoding step, but it nevertheless remains the primary bottleneck of QEC.


Directly competing with speed is accuracy. If a decoder is inaccurate, errors will be missed or introduced each QEC round and will propagate to ruin the computation. High-distance codes are necessary for accuracy, but unfortunately introduce high-qubit overheads and make decoding much more challenging. Advances in QEC code design and low-latency integration between AI supercomputers and QPUs alleviate pressure on the decoding step, but it nevertheless remains the primary bottleneck of QEC.



## Pauli Frames and Error Tracking

In practice, when errors are identified by the decoder, they are not immediately corrected but are tracked using a **Pauli frame**. The Pauli frame keeps track of the corrections classically and applies them later. This approach reduces the number of gate operations required to fix errors on the QPU, thereby protecting the encoded state from additional noise introduced by each correction gate. For instance, if a bit flip error occurs on qubit 1 in the first round and another bit flip error happens on the same qubit later, the two errors cancel each other out, eliminating the need for a correction

Often, codes are depicted using 3D images like the one below. In this case, each plane is a Steane code QEC round with flagged errors in purple. Each error is saved, and the list grows with future rounds. The final Paul frame, $[X_1, X_5, X_1]$, is the list of corrections for the three bit flip errors that have occurred over all the rounds: two on qubit 1 and one on qubit 5.  In the last step, the errors can be simplified, for example, $X_1X_1 = I$, so only one of the three corrections, $X_5$, needs to be applied. This is a rather trivial example, and often diagrams like this are used to depict more complex codes and their respective error pathways.

<img src="https://github.com/osbama/KBM608/blob/main/hands-on/hands-on-7-images/pauliframes.png?raw=1" alt="3D visualization of Pauli frame tracking across multiple Steane code QEC rounds, showing purple error markers on successive planes and the accumulated Pauli frame corrections" style="width: 700px;"/>

The dimension of time can also lend itself to more sophisticated decoding schemes. This is particularly important when measurement errors occur during the stabilizer checks. In this case, it might appear that a stabilizer flags when in fact the data qubits are fine. Multiple decoding rounds can demonstrate that the false stabilizer flag is a consequence of measurement error and not a true error, where other true errors would persist without correction. Such an approach is more powerful but requires decoding of much more complex syndromes. The diagram below demonstrates this concept with an example.

<img src="https://github.com/osbama/KBM608/blob/main/hands-on/hands-on-7-images/decodeintime.png?raw=1" alt="Diagram comparing single-round decoding versus decoding over multiple time steps, showing how measurement errors can be detected when syndrome data from consecutive rounds is analyzed together" style="width: 1000px;"/>
Notice how, in the first case, the decoder has kept track of a measurement error and is therefore making an incorrect syndrome in the final case. When decoding happens over time, the decoding task must not decode a 19-bit syndrome but is able to flag measurement errors



### Exercise 1:

The benefit of decoding in time is that the measurement errors can be factored into the decoding process.  However, the tradeoff is that the decoding problem is much harder.  When decoding in time, an effective parity check matrix must be constructed as an input to the decoder. In this exercise you will build $H^{(2)}$ for a two round Steane code that includes consistency checks to  flag errors between the two time steps.  

First, a few hints.  Consider the dimensions. The number of columns still corresponds to the number of qubits, but, now we need to take into account the data qubits at time 0, the data qubits at time 1, and the three ancilla qubits used to measure syndromes between the two rounds. 

Each time step will have the same three stabilizer checks, so $H^{(2)}$ should have six rows. 

The top left block of $H^{(2)}$ and the bottom right block will be $H$, where $H$ is the standard Steane code parity check matrix.

What do the middle three columns need to be for $H^{(2)}$ to be able to catch measurement errors?

Build $H^{(2)}$, and then build an error vector $e$ of size 17 such that each entry is a 0 or a 1 if an error occurred on that qubit.  Compute $H^{(2)}e^T$ for a case with an error on data qubit 1 in the first time step, an error on data qubit 1 in the second time step only, and a measurement error. Note, it is best practice to assume that the decoder will not hanndle raw syndrome outputs, but the differences between he current set of measurements and the next round.  For example, after preparation the syndrome 101 might be measured. If the next round produces the same stabilizer measurerments, the decoder would see 000 not 101. This syntax makes it much easier for decoders to handle data in more complex settings.



In [ ]:
# EXERCISE 1
H = np.array([
    [1, 1, 0, 1, 1, 0, 0],
    [1, 0, 1, 1, 0, 1, 0],
    [0, 1, 1, 1, 0, 0, 1]
])

# Build 2 round parity check matrix.
H2 = np.array([##TODO##
])

# Syndrome for no error
e = np.array([##TODO##])

print(H2 @ e.T)

# Syndrome for error on first data qubit in first time step
e = np.array([##TODO##])

print(H2 @ e.T)

# Syndrome for error on first measurement qubit
e = np.array([##TODO##])

print(H2 @ e.T)

# Syndrome for error on first data qubit in second time step
e = np.array([##TODO##])

print(H2 @ e.T)

# Try other errors

Looking at your results. Can you see how a measurement error can be detected in the the symdrome pattern? Note that the parity check matrix you created can only catch measurment errors on round 1.  A round 2 measurement error would be missed unless you extenced the parity check matrix further.  

## Most Likely Error Decoding

So far, decoders have been presented as black boxes. In many cases that is sufficient.  If you are developing or testing new codes, you might just use a state of the art decoder and not care how it works.  In other cases, the opposite is true, and you might focus on developing and tuning a decoder to work well for a specific sort of QEC situation. 

The rest of this lab will allow you to explore a number of different decoders ranging from conceptually simple to state of the art.  You will interact with them in different ways, in certain cases you'll write the decoder from scratch and at other times you'll use the decoder out of the box. 

The starting point is to consider a naive brute forced decoder that is conceptually simple yet sets the upper bound on decoder accuracy.  

The steps of **maximum likelihood decoding** are as follows (considering only bitflip errors for simplicity):

1. Select a QEC code and encode a message in the codespace with $n$ data qubits.
2. Generate the $2^n$ bitstrings $\{x_0, \cdots, x_{2^n} \}$ of length $n$ corresponding to all possible error situations.
3. For each $x_i$, compute the syndrome as $Hx_i~\mathrm{mod} 2$ where $H$ is the code's parity check matrix.
4. Then, for each possible syndrome, list errors that could have produced the given syndrome.
5. Under each syndrome, order errors by their Hamming distance from the original message.
6. Finally, the decoder will receive a new syndrome, and then select the lowest Hamming distance error and apply the fix.

The assumption here is that, if errors are independent, cases with fewer errors are always more likely than those with more errors. 

Notice, in Lab 2 when you coded the Steane code, we assumed a situation where one error occurs at a time, allowing your syndrome checks to fix errors.  This is the same assumption made here. The problem with this approach is that it does not scale.  There are $2^n$ errors that need to be computed *a priori* which is not possible for large codes.  The sections below will consider more scaleable heuristics to overcome this issue.



 

### Exercise 2:

Code the most likely error decoder for the Steane code below given the parity check matrix below.  For each syndrome, print the associated length 7 bitstrings that produce that error, the Hamming distance from the baseline message (0000000), and the probability of that error.



In [ ]:
# EXERCISE 2
# Define the bit-flip probability
p = 0.1

# Define the parity check matrix H
H = np.array([
    [1, 1, 0, 1, 1, 0, 0],
    [1, 0, 1, 1, 0, 1, 0],
    [0, 1, 1, 1, 0, 0, 1]
])

# Generate the syndromes and sort based on the instructions above
##TODO##

## AI Decoders

One way to circumvent the scaling challenges posed by the a brute force most likely error decoder is to use tools like AI. AI is fantastic at pattern recognition, runs very quickly, and can easily scale. 

AI decoders also offer flexibility as they can be trained with simulated data or even trained on small-distance codes and, via transfer learning, be used to decode higher distance codes.

Recently, [NVIDIA and QuEra announced a new transformed based decoder](https://developer.nvidia.com/blog/nvidia-and-quera-decode-quantum-errors-with-ai/) tested on magic state distillation circuits used by QuEra (A 35 qubit circuit with 5 Steane code logically encoded logical qubits).  The decoder showed promise by outperforming the decoder used by QuEra in terms of speed and accuracy.  Additionally, the AI decoder might have the potential to scale to code distances large enough for sufficiently low logical error rates.

<img src="https://github.com/osbama/KBM608/blob/main/hands-on/hands-on-7-images/aidecoderplot.png?raw=1" alt="Performance comparison plot of the NVIDIA-QuEra AI transformer decoder versus baseline, showing improved accuracy and speed on magic state distillation circuits" style="width: 700px;"/>

### Exercise 3:

You will now build a working AI decoder for the Steane code. The goal is to build something similar to the workflow in the image below.

<img src="https://github.com/osbama/KBM608/blob/main/hands-on/hands-on-7-images/aidecoderworkflow.png?raw=1" alt="Workflow diagram showing the AI decoder pipeline: CUDA-Q generates noisy syndrome data, which is split into training and test sets to train a neural network that predicts logical errors" style="width: 700px;"/>

This lab does not expect you to have experience coding AI models with tools like PyTorch, so you will focus on the data generation and learn how to prepare the data to train an AI decoder without worrying about details of the model. Follow the steps outlined below to complete the code.



The first step is to generate the training data. Take the Steane code circuit you coded in Lab 2, now with bitflip noise to each qubit after encoding.  In this case, we can explore circuit-level noise based on simulated results rather than a contrived data set. 

Create a data set of 5000 samples.  To generate this, run `cudaq.run()` 5000 times taking one shot each time.  Output the measurements from the syndrome checks (without correcting any errors) and then measure all of the data qubits.  Compute the parity of bits corresponding to the correct logical operator to determine the true logical state.  

Save the syndromes and the logical states as two numpy arrays.  This will be your data set.

In [ ]:
# EXERCISE 3
p = 0.05
cudaq.unset_noise()
noise = cudaq.NoiseModel()

@cudaq.kernel
def steane_code() -> list[int]:
    """Prepares a kernel for the Steane Code
    Returns
    -------
    cudaq.kernel
        Kernel for running the Steane code
    """   
    data_qubits = cudaq.qvector(7)
    ancilla_qubits = cudaq.qvector(3)

    # Create a superposition over all possible combinations of parity check bits
    h(data_qubits[4])
    h(data_qubits[5])
    h(data_qubits[6])

    #Entangle states to enforce constraints of parity check matrix

    x.ctrl(data_qubits[0],data_qubits[1])
    x.ctrl(data_qubits[0],data_qubits[2])

    x.ctrl(data_qubits[4],data_qubits[0])
    x.ctrl(data_qubits[4],data_qubits[1])
    x.ctrl(data_qubits[4],data_qubits[3])

    x.ctrl(data_qubits[5],data_qubits[0])
    x.ctrl(data_qubits[5],data_qubits[2])
    x.ctrl(data_qubits[5],data_qubits[3])

    x.ctrl(data_qubits[6],data_qubits[1])
    x.ctrl(data_qubits[6],data_qubits[2])
    x.ctrl(data_qubits[6],data_qubits[3])

    for j in range(7):
        cudaq.apply_noise(cudaq.XError, p, data_qubits[j])

    # Detect X errors
    h(ancilla_qubits)

    z.ctrl(ancilla_qubits[0],data_qubits[0])
    z.ctrl(ancilla_qubits[0],data_qubits[1])
    z.ctrl(ancilla_qubits[0],data_qubits[3])
    z.ctrl(ancilla_qubits[0],data_qubits[4])

    z.ctrl(ancilla_qubits[1],data_qubits[0])
    z.ctrl(ancilla_qubits[1],data_qubits[2])
    z.ctrl(ancilla_qubits[1],data_qubits[3])
    z.ctrl(ancilla_qubits[1],data_qubits[5])

    z.ctrl(ancilla_qubits[2],data_qubits[1])
    z.ctrl(ancilla_qubits[2],data_qubits[2])
    z.ctrl(ancilla_qubits[2],data_qubits[3])
    z.ctrl(ancilla_qubits[2],data_qubits[6])

    h(ancilla_qubits)

    d0=mz(data_qubits[0])
    d1=mz(data_qubits[1])
    d2=mz(data_qubits[2])
    d3=mz(data_qubits[3])
    d4=mz(data_qubits[4])
    d5=mz(data_qubits[5])
    d6=mz(data_qubits[6])
    a0=mz(ancilla_qubits[0])
    a1=mz(ancilla_qubits[1])
    a2=mz(ancilla_qubits[2])

    return [d0,d1,d2,d3,d4,d5,d6,a0,a1,a2]

# Generate Data
##TODO## - generate the inout data 

raw_logical_error_rate =##TODO## - calculate the raw logical error rate

The previous cell is quite a bit of work, and requires you to manually construct the entire QEC code. This is not ideal when you are primarily interested in testing an AI decoder and want data generation streamlined.

Another more efficient way to sample training data for memory experiments is directly using the parity check matrix. Random bitflips can be applied and syndromes determined via matrix multiplication. CUDA-Q QEC can do this with just a few lines of codes (shown below) and generate the same sort of data we did above with a preloaded Steane code. Additionally, if you want to generate data for multiple syndrome extraction rounds, you can use the sample_memory_circuit. If you want to test a new, non-standard code, you would need to define the kernels explicitly similar to the example above as shown in the docs [here](https://nvidia.github.io/cudaqx/components/qec/introduction.html#qec-code-framework-cudaq-qec-code)

In [ ]:
steane = qec.get_code("steane")

Hz = steane.get_parity_z()

observable = steane.get_observables_z()

data = qec.generate_random_bit_flips(Hz.shape[1], p)

syndrome = Hz @ data % 2
print(f"syndrome: {syndrome}")

actual_observable = observable @ data % 2
print(f"actual_observable: {actual_observable}")

Your data set will be split into a 4000 point training set and a 1000 point test set to validate that the model worked.  The inputs to the model will be syndrome data obtained from your simulations.  The loss function will be a comparison between the predicted logical state of the model and the logical state you obtained from measuring the data qubits. 

If you defined the arrays properly, the code below should preprocesses the data for you, splitting it into a training and test set, construct a simple neural network, define model setting, and train the model.

Each epoch is one trining loop through the entire data set.  The model will then test the accuracy using the test set which was excluded from training at each epoch. The code is provided for the interested reader to see the details of the torch implementation, can also just be run as a black box. Do the plots below indicate the model improved as training progressed?

In [ ]:

# This section normalizes and loads the data you defined previously
syndromes = np.array(syndromes, dtype=np.float32)
logical_flips = np.array(logical_flips, dtype=np.float32)

# Normalize input data
syndromes = (syndromes - syndromes.mean()) / syndromes.std()

X_train, X_test, y_train, y_test = train_test_split(
    syndromes, logical_flips, test_size=0.20, random_state=42)

X_train = torch.tensor(X_train)
X_test = torch.tensor(X_test)
y_train = torch.tensor(y_train)
y_test = torch.tensor(y_test)

# Create data loaders
batch_size = 32
train_loader = torch.utils.data.DataLoader(
    torch.utils.data.TensorDataset(X_train, y_train),
    batch_size=batch_size,
    shuffle=True
)

# This builds a simple NN model which will be trained
class SimpleDecoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(3, 32)
        self.fc2 = nn.Linear(32, 16)
        self.fc3 = nn.Linear(16, 1)
        self.act = nn.Sigmoid()
        
        # Initialize weights
        nn.init.xavier_uniform_(self.fc1.weight)
        nn.init.xavier_uniform_(self.fc2.weight)
        nn.init.xavier_uniform_(self.fc3.weight)

    def forward(self, x):
        x = torch.relu(self.fc1(x))
        x = torch.relu(self.fc2(x))
        return self.act(self.fc3(x))

model = SimpleDecoder()
criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.1, patience=10,)

# Define the test_accuracy function
def test_accuracy():
    model.eval()
    with torch.no_grad():
        pred = (model(X_test).squeeze() > 0.5).float()
        return (pred == y_test).float().mean().item()

# Training loop
num_epochs = 30
train_losses, test_acc = [], []

# epoch 0 (untrained)
test_acc.append(test_accuracy())

for epoch in range(1, num_epochs + 1):
    model.train()
    epoch_loss = 0
    batch_count = 0
    
    # Training
    for batch_X, batch_y in train_loader:
        optimizer.zero_grad()
        out = model(batch_X).squeeze()
        loss = criterion(out, batch_y)
        loss.backward()
        optimizer.step()
        
        epoch_loss += loss.item()
        batch_count += 1
    
    # Calculate average loss for the epoch
    avg_epoch_loss = epoch_loss / batch_count
    train_losses.append(avg_epoch_loss)
    
    # Evaluate accuracy
    current_acc = test_accuracy()
    test_acc.append(current_acc)
    
    # Update learning rate
    scheduler.step(avg_epoch_loss)
    
    if epoch % 1 == 0:
        print(f"Epoch {epoch:3d}/{num_epochs} | "
              f"train loss={avg_epoch_loss:.4f} | "
              f"test acc={current_acc:.4f}")

# Plotting
plt.figure(figsize=(12,4))

plt.subplot(1, 2, 1)
plt.plot(range(num_epochs + 1), test_acc, marker='o')
plt.xlabel("Training Epoch")
plt.ylabel("Test Accuracy")
plt.title("Steane-Code Decoder Accuracy")
plt.grid(True)
plt.ylim(0, 1)

plt.subplot(1, 2, 2)
plt.plot(range(1, num_epochs + 1), train_losses, color='red')
plt.xlabel("Training Epoch")
plt.ylabel("Training Loss")
plt.title("Training Loss Over Time")
plt.grid(True)

plt.tight_layout()
plt.show()

# Print final metrics
print(f"\nFinal Results:")
print(f"Final Test Accuracy: {test_acc[-1]:.4f}")
print(f"Final Training Loss: {train_losses[-1]:.4f}")
print(f"Raw (Undecoded) Accuracy: {1-raw_logical_error_rate:.4f}")

# Print first 20 predictions
model.eval()
with torch.no_grad():
    test_predictions = model(X_test).squeeze()
    predicted_labels = (test_predictions > 0.5).float()

print("\nFirst 20 test examples:")
print("=" * 40)
print(f"{'Index':^6} | {'True':^6} | {'Predicted':^6}")
print("=" * 40)

for i in range(20):
    true_label = int(y_test[i].item())
    pred_label = int(predicted_labels[i].item())
    print(f"{i:^6} | {true_label:^6} | {pred_label:^6}")

print("=" * 40)

# Calculate accuracy for these 20 examples
correct = (predicted_labels[:20] == y_test[:20]).sum().item()
print(f"\nAccuracy for these 20 examples: {correct}/20 = {correct/20:.2%}")



You should see the model successfully train! The test set accuracy should increase while the loss functions decreases.

There are a few key observations to discuss.

1. If the model parameters are random and we run this training multiple times, we should see the model on average start with an accuracy of about 0.5. This means we would have as much luck flipping a coin as our decoder. It may start higher or lower depending on the initial parameters bias towards outputting 1's or 0's. So, we do demonstrate the model did train.

2. The trained model does outperform the raw logical error rate without decoding. So, our AI decoder is an improvement. Given the simplicity of the Steane code, this is unsurprising, as there is not really any hidden insight to be gleaned as we can essentially work out the brute force MLE decoding by hand.

3. The final output of the AI model (or any decoder) is limited by the underlying QEC code and its distance. This is what determines what errors are detectable or correctable before we even try decoding. For example, the distance three Steane code cannot correct two errors. So, we cannot expect the AI model to learn how to correct these errors either. Thus, AI decoding shines in the regime where there are many correctable errors with non-trivial syndrome patterns.

4. Because the trained decoder depends on the error model used, it is really important to have large training data sets with sufficiently realistic noise models and model often require fine tuning with experimental data to realize peak performance. If the error rate was tiny, the model may just learn to output logical 0 all the time, and learn nothing about the error patterns if it has insufficient cases to train on. When training on physical QPU data, we are not trying to learn a noise model, but the actual, unknown noise profile of the device.

## Belief Propagation Decoding

Another state-of-the-art decoding method is **belief propagation (BP)**.  BP is a powerful technique borrowed from classical error correction that is highly flexible and can serve as a black box decoder for arbitrary QEC Codes. It is particularly useful for codes like quantum low-density parity check (**qLDPC**). All the user needs to do is provide a parity check matrix and then feed the decoder syndromes to decode. 

NVIDIA created a GPU accelerated BP decoder which allows researchers to push QEC even further than before.  This section will walk you through implementing BP and how to use NVIDIA's accelerated BP decoder. 

At a high level, BP works by taking initial beliefs about each data qubit's physical error rate and, using syndrome measurements, iteratively update these beliefs to converge on a solution which provides a likelihood of an error occurring or not.  

Though there are different variations of BP, one implementation is the following:
1. Initialize a log-likelihood ratio $L_{v_i} = log(\frac{1-p}{p})$ for each data qubit $v_i$ where $p$ is the physical error rate.  If an error is unlikely, the quantity is positive and increases in magnitude the smaller $p$ is as errors are less likley.
   
2. $L_{v_i\rightarrow c_j}$, the message sent from each data qubit to a check qubit, is calculated based on the following equation:  $L_{v\rightarrow c} = L_i + \sum_{k \in N(i)/j} L_{c_k\rightarrow v_i}$ where $\sum_{k \in N(i)/j}$ includes all of the check qubits connected to data qubit $v_i$ excluding check qubit $c_j$. That is, the check qubit is updated with beliefs the variable learned from the *other* check qubits it is connected to.

   
3. Similarly, $L_{c_j\rightarrow v_i}$ is computed to compute the messages sent back to the variable qubits using $L_{c_j \to v_i} = (-1)^{s_j} \cdot 2 \cdot \text{arctanh} \left( \prod_{k \in N(j) \setminus i} \tanh \left( \frac{L_{v_k \to c_j}}{2} \right) \right)$ where $N(j) \setminus i$ is all *other* variable nodes connected to $c_j$ excluding $v_i$ and $s_j$ is the value of the syndrome measurement for $c_j$.
   
4. Finally, the final beliefs $L_{\text{final}, i}$ are computed as $L_i + \sum_{j \in N(i)} L_{c_j \to v_i}$, summing the prior beliefs with the final messages sent to each variable node. From this a decision can be made where positive numbers indicate no error and negative an error, with the magnitudes related to confidence.

Ideally, BP will converge to a solution that agrees with the original syndrome and correct the error.  If BP cannot converge, it means there is still significant uncertainty whether some of the bits have errors or not and postprocessing is necessary to refine the result.  This will be discussed in the following section.


### Exercise 4:

Below is the start of a BP implementation for decoding the 5-qubit repetition code. Fill in the sections marked "TODO" to complete the code.  Most of the BP loops are calculated for you.  Make sure to review them and understand what is going on.  Then, you will complete the code by fixing the code to calculate the final belief on each qubit and determine where errors occurred.


In [ ]:
# EXERCISE 4
physical_error_rate = 0.1
max_iter = 5

H = np.array([
    [1, 1, 0, 0, 0],  
    [0, 1, 1, 0, 0],  
    [0, 0, 1, 1, 0],  
    [0, 0, 0, 1, 1]   
], dtype=int)

n_qubits = H.shape[1]
n_checks = H.shape[0]

actual_error = np.array([0, 0, 1, 0, 0])

syndrome = H @ actual_error % 2

print("--- Simulation Setup ---")
print(f"Physical Error Rate (p): {physical_error_rate}")
print(f"Actual Error Vector:     {actual_error}")
print(f"Resulting Syndrome:      {syndrome}")
print("-" * 40 + "\n")


#Initialize the prior probabilites, and matricies for v -> c and c -> v messages
L_prior = np.log((1 - physical_error_rate) / physical_error_rate)

L_v_to_c = np.zeros((n_qubits, n_checks))

L_c_to_v = np.zeros((n_checks, n_qubits))


print("--- Initialization ---")
print(f"Prior LLR (L_i) for all qubits: {L_prior:.4f}\n")



for iteration in range(max_iter):
    print(f"--- Iteration {iteration + 1} ---")


    # Compute L_{v_i -> c_j} = L_i + sum_{k in N(i)\j} L_{c_k -> v_i}
    # The message is the qubit's own belief (prior) plus all incoming messages from checks
    
    for i in range(n_qubits):
        for j in range(n_checks):
            if H[j, i] == 1:
                connected_checks = np.where(H[:, i] == 1)[0]
                sum_incoming_L = 0
                for k in connected_checks:
                    if k != j:
                        sum_incoming_L += L_c_to_v[k, i]
                L_v_to_c[i, j] = L_prior + sum_incoming_L
    
    print("Step A: Updated Variable-to-Check Message Matrix (L_v_to_c):")
    print(np.round(L_v_to_c, 4))
    print("")


    # Compute L_{c_j -> v_i} = (-1)^s_j * 2 * atanh( product_{k in N(j)\i} tanh(L_{v_k -> c_j}/2) )
    # The message is based on the check's syndrome value and all of the incoming messages from the data qubits.

    for j in range(n_checks):
        for i in range(n_qubits):
            if H[j, i] == 1:
                connected_qubits = np.where(H[j, :] == 1)[0]
                prod_tanh = 1.0
                for k in connected_qubits:
                    if k != i:
                        # tanh can be unstable for large LLRs, so we clip its argument
                        val = np.clip(L_v_to_c[k, j] / 2, -20, 20)
                        prod_tanh *= np.tanh(val)

                sign = (-1)**syndrome[j]
                
                # FIX IS HERE: Clip prod_tanh to avoid arctanh(1) or arctanh(-1)
                # This defines the missing variable 'prod_tanh_clipped'
                epsilon = 1e-12
                prod_tanh_clipped = np.clip(prod_tanh, -1 + epsilon, 1 - epsilon)

                L_c_to_v[j, i] = sign * 2 * np.arctanh(prod_tanh_clipped)



    print("Step B: Updated Check-to-Variable Message Matrix (L_c_to_v):")
    print(np.round(L_c_to_v, 4))
    print("")


##TODO## Compute the final belief of each qubit (prior beliefs plus sum of incoming messages)


##TODO## END

print("--- Final Belief ---")
print(f"Final LLRs: {np.round(L_final, 4)}")


estimated_error = ##TODO## Determine where errors occurred based on L_final

print("-" * 40)
print(f"Decoder Estimated Error:  {estimated_error}")

if np.array_equal(actual_error, estimated_error):
    print("\n✅ Success: The decoder correctly identified the error.")
else:
    print("\n❌ Failure: The decoder did not find the correct error.")

The 5-qubit repetition code is small, and easy to decode.  If more than one error is present, the BPO decoder will fail, as the code distance is the primary constraint. However, with more complex codes, BP may complete, but not converge, meaning uncertainty remains on some of the bits and a logical error may occur due to its failure. Also, decoding large codes with orders of magnitude more variable and check nodes can become cumbersome. 

NVIDIA's accelerated BP decoder, now part of the CUDA-Q QEC library, helps address this problem and parallelizes the BP algorithm.  Examining your code above, it is not hard to see how the message calculations can be trivially parallelized, making them well suited for GPU acceleration. 

Try running NVIDIA's accelerated BP decoder by running the code below which imports a data set of syndrome data from a large qLDPC code.  How long did it take to decode 10 shots?  


In [ ]:
if __name__ == "__main__":
    # See other test data options in https://github.com/NVIDIA/cudaqx/releases/tag/0.2.0
    filename = 'osd_1008_8785_0.001.json' # lower error rate
    bz2filename = filename + '.bz2'
    if not os.path.exists(filename):
        url = f"https://github.com/NVIDIA/cudaqx/releases/download/0.2.0/{bz2filename}"
        # Download the file
        response = requests.get(url, stream=True)
        response.raise_for_status()  # Raise an error if download fails
        with open(bz2filename, "wb") as f:
            for chunk in response.iter_content(chunk_size=8192):
                f.write(chunk)

        print(f'Decompressing {bz2filename} into {filename}')

        # Decompress the file
        with bz2.BZ2File(bz2filename, "rb") as f_in, open(filename,
                                                          "wb") as f_out:
            f_out.write(f_in.read())

        print(f"Decompressed file saved as {filename}")

In [ ]:
if __name__ == "__main__":
    num_shots = 10
    run_as_batched = True
    print_output = True
    osd_method = 0 # 0 is off 1 is on
    run_decoder(filename, num_shots, run_as_batched, print_output, osd_method)

One of the reasons this algorithm can be accelerated is that it can be naturally parallelized. Do you see where that might happen?  

The answer lies in the process of sending messages from syndrome qubits to data qubits, and vice versa. Each of these messages can be computed in parallel. This corresponds to performing operations that alternate across the rows and columns of a matrix, depending on which message is being sent. Each row and column can be treated independently, allowing for further parallelization within each row or column.

In benchmarks of large code instances, the NVIDIA decoder was up to 35x faster than the industry standard implementation for benchmarks run on the [[144,12,12]](https://arxiv.org/abs/2308.07915) code. 

<img src="https://github.com/osbama/KBM608/blob/main/hands-on/hands-on-7-images/benchmark.png" alt="Bar chart showing NVIDIA GPU-accelerated BP decoder performance benchmarks, demonstrating up to 35x speedup over industry standard implementations on the [[144,12,12]] qLDPC code" style="width: 700px;"/>



### Batching

The performance can be enhanced ever further using batching.  Batching means that a collection of syndromes are sent to the GPU and decoded simultaneously.  This minimizes data transfer and allows the entire GPU to be working, reducing the average decoding time per shot.  

Run the cells below to see the impact of batching when decoding 10000 syndromes. In the second cell, `run_as_batched` is set to `True`. How much faster is batching?  Batching is most effective when decoding a large number of syndromes.

In [ ]:
if __name__ == "__main__":
    num_shots = 10000
    run_as_batched = False
    print_output = False
    osd_method = 0 # 0 is off 1 is on
    run_decoder(filename, num_shots, run_as_batched, print_output, osd_method)

In [ ]:
if __name__ == "__main__":
    num_shots = 10000
    run_as_batched = True
    print_output = False
    osd_method = 0 # 0 is off 1 is on
    run_decoder(filename, num_shots, run_as_batched, print_output, osd_method)

### Ordered Statistics Decoding
Now, increase the error rate using a different data file from 0.1 % to 0.5 %.  What happens to the logical error rate? 

In [ ]:
if __name__ == "__main__":
    # See other test data options in https://github.com/NVIDIA/cudaqx/releases/tag/0.2.0
    filename = 'osd_1008_8785_0.005.json' # lower error rate
    bz2filename = filename + '.bz2'
    if not os.path.exists(filename):
        url = f"https://github.com/NVIDIA/cudaqx/releases/download/0.2.0/{bz2filename}"
        # Download the file
        response = requests.get(url, stream=True)
        response.raise_for_status()  # Raise an error if download fails
        with open(bz2filename, "wb") as f:
            for chunk in response.iter_content(chunk_size=8192):
                f.write(chunk)

        print(f'Decompressing {bz2filename} into {filename}')

        # Decompress the file
        with bz2.BZ2File(bz2filename, "rb") as f_in, open(filename,
                                                          "wb") as f_out:
            f_out.write(f_in.read())

        print(f"Decompressed file saved as {filename}")

In [ ]:
if __name__ == "__main__":
    num_shots = 10000
    run_as_batched = True
    print_output = False
    osd_method = 0 # 0 is off 1 is on
    run_decoder(filename, num_shots, run_as_batched, print_output, osd_method)

You should notice a very high logical error rate. Each of these logical error rates is a case where BP could not converge.  Like any numerical solver, it is possible to increase iterations and perform a number of tricks to help BP, but some particularly difficult syndromes will not converge either way. To solve this, BP decoders are often paired with optional ordered statistics decoding (OSD) which performs post-processing to improve the result.  Thus, you might often see a "BP+OSD" decoder. 

The intuition behind OSD decoding is to solve for the errors directly using matrix inversion.  That is, solving $e = H^{-1}s$ where $e$ is a vector with error locations, $s$ is the syndrome, and $H$ is the parity check matrix.  The procedure cannot be performed with the original $H$, so instead, a square matrix with columns of full-rank must be constructed from the columns of $H$. 

What makes this approach challenging is selecting the right columns.  Many different solutions can be obtained with different column selections, but without guidance, this will likely result in a solution that is not of minimum weight and induce a logical error. Recall, for the most likely error decoder above, that valid errors with high weight are much less likely than low weight errors.  Same idea here, but rather than computing all combinations, the OSD heuristic provides a clever way to search from a really good starting point that is the output of BP. 

The output of BP assigns a probability of error to each data qubit.  This naturally informs the reordering of the columns to rank from most to least likely to have an error. Next Gaussian elimination is performed on $H$ to determine the first full columns of full rank. These subcolumns are then inverted to get $H^{-1}$ which is multiplied by $s$ to get the error result $e_{[s]}$. The $e_{[s]}$ is then padded with zeros  $e_{[T]} =0$ which results in a total error profile of $ e = (e_{[s]},e_{[T]})$.  


### Exercise 5:

The following exercise is based on a [lecture](https://www.youtube.com/watch?v=b9N2Ps3FTto) by Joschka Roffe. Given the parity check matrix below and the probabilities of error from BP. Perform OSD manually and find the error profile that satisfies the syndrome. Note, all computations must be performed using mod 2 arithmetic. This can be accomplished using the `galois` library which creates a Galois field and allows all `numpy` operations to compute mod 2.


In [ ]:
# EXERCISE 5
GF2 = galois.GF(2) # allows mod 2 math.

#parity check matrix as numpy array
H = np.array([[0, 1, 0, 0, 1, 0],
              [0, 0, 1, 0, 0, 1],
              [1, 0, 0, 1, 0, 0],
              [1, 0, 0, 0, 1, 1]])

bp_results = [0.05, 0.5, 0.01, 0.01, 0.82, 0.05]

s = GF2([1,0,0,1]) #syndrome

# Get indices that would sort bp_results in descending order
##TODO##

# Rearrange columns of numpy array
##TODO##
H_sorted = 


# Convert to GF2 matrix after rearrangement for mod 2 aritmatic
GF2 = galois.GF(2)
H_sorted = GF2(H_sorted)

# Perform Gaussian elimination to identify the first four columns of full rank.
def rref(matrix):
    rows, cols = matrix.shape
    r = 0
    for c in range(cols):
        if r >= rows:
            break
        if matrix[r, c] == 0:
            for i in range(r + 1, rows):
                if matrix[i, c] != 0:
                    matrix[[r, i]] = matrix[[i, r]]
                    break
        if matrix[r, c] == 0:
            continue
        matrix[r] = matrix[r] / matrix[r, c]
        for i in range(rows):
            if i != r and matrix[i, c] != 0:
                matrix[i] = matrix[i] - matrix[i, c] * matrix[r]
        r += 1
    return matrix

print("Gaussian Elimination Result")
print(rref(H_sorted.copy())) # First four columns are pivot columns

# Build H_s from the first full rank columns
##TODO##

# Compute Hs_inverse
##TODO##

# Calculate e_s
##TODO##

# Pad result with zeros and reorder based on colum sorting from earlier.
##TODO##

# Confirm that the errors produce the expected syndrome from the original H
##TODO##

from Images.decoder.solution3_button import show_cudaq_solution
show_cudaq_solution()

OSD guarantees an error pattern that satisfies the syndrome, but it is not necessarily a minimum weight solution nor avoids a logical error. In the exercise above, you performed OSD-0, or zero order OSD.  To improve results, higher order OSD can be performed.  Higher order OSD performs bitflips in the $e_{[T]}$ bits and checks if a lower weight solution can be found. Generally speaking, the order of the OSD determines how many bitflips are considered, but there is additional nuance not covered here. 

Lower weight solutions are possible as bit flips in $e_{[T]}$ also impact the other bits in the $e_{[s]}$ subspace. 

To summarize, syndromes are of varying difficulty. Easy syndromes are solved with BP, OSD-0 is used for moderate syndrome difficulties, and higher order OSD is used for the most challenging.  


<img src="https://github.com/osbama/KBM608/blob/main/hands-on/hands-on-7-images/bposd.png?raw=1" alt="Flowchart illustrating the BP+OSD decoding pipeline: BP attempts to converge on easy syndromes, OSD-0 handles moderate cases, and higher-order OSD processes the most challenging syndromes" style="width: 1000px;"/>


Try running the code below on the 10000 shot data set.  See what happens when `osd_method` is set to 1 for OSD-0.  Then try setting this variable to 3 to run a variant of higher order OSD.  Does the logical error rate improve?  How much more time does it take to perform higher order OSD?

In [ ]:
if __name__ == "__main__":
    num_shots = 10000
    run_as_batched = True
    print_output = False
    osd_method = 1  # 0 is off, 1 OSD-0, 3 is OSD-X, where X is the order
    run_decoder(filename, num_shots, run_as_batched, print_output, osd_method)

In [ ]:
if __name__ == "__main__":
    num_shots = 10000
    run_as_batched = True
    print_output = False
    osd_method = 3 # 0 is off, 1 OSD-0, 3 is a higher order OSD
    osd_order = 1
    run_decoder(filename, num_shots, run_as_batched, print_output, osd_method, osd_order)

# Decoder Metrics and (Temporal) Parallel Window Decoding 

As QEC matures as a field, research is expanding from its origins in pure theory to experimental demonstrations of QEC workflows on physical QPUs. As QPUs continue to scale, it is becoming increasingly important to consider how every aspect of a QEC workflow scales such that it is possible to run fault-tolerant logic on devices with millions of qubits.

Decoders are likely the primary bottleneck of QEC, so it becomes critically important to understand the nature of these bottlenecks and what sorts of solutions will scale or not.

This notebook will explore the key metrics of decoders, giving you a better intuition for some of the most important practical considerations for decoders and why they matter. It will also explore a scalable decoding scheme called temporal parallel window decoding which has promise to help overcome some of the challenges faced with QEC decoding.


> **Note:** Run the cell below to import all required packages.
> If you installed packages above, restart the kernel first
> (**Runtime → Restart session** in Colab, or **Kernel → Restart** in Jupyter).

In [ ]:
from Images.parallel.decoder_simulator import QECSimulator

## The Key Decoder Metrics

Recall that the job of a decoder is to take syndromes measured from a quantum computer and a parity check matrix ($H$), defining the parity constraints of the QEC code, and determine where errors occurred so they can be fixed. Decoding a single syndrome extraction round can reveal errors in space, but certain errors, such as measurement errors, can only be caught when decoding in time, using a larger $H$ that can decode multiple rounds of syndrome data.

Decoders can be assessed by three primary metrics:

1. **Accuracy** - The ability to correctly identify errors.
2. **Throughput** - The rate at which the decoder processes syndrome data.
3. **Reaction time** - The time between when the last syndrome is sent from the QPU and the decoder returns a correction.

Accuracy is fairly straightforward. If a decoder makes poor predictions, logical errors occur which results in poor outcomes from the quantum algorithm. This is usually measured by a logical error rate. We will not discuss accuracy much here, but note that there is often a tradeoff between decoder accuracy and its scalability with respect to the number of syndromes decoded in a single block.

Throughput measures how fast syndromes can be processed by the decoder. This rate ($r_{proc}$) must be faster than the rate at which syndromes arrive from the quantum computer $r_{gen}$. If syndromes arrive faster than they are processed, a backlog starts to build up and the decoder grinds to a halt.

Consider an example of the steady-state intercircuit feed-forward latency (SIFL) benchmark shown below. It begins with a preparation of the logical 1 state. Then, 10 syndrome extraction rounds are performed before a measurement occurs and a feed-forward operation determines if an $X$ gate needs to be applied or not before the next measurement.

Between the grey and purple boxes, 10 syndrome extraction rounds occur to flag a bitflip error(s) that might have occurred and potentially induce a logical error when measured. The syndromes must be decoded before the decision is made to apply the $X$ gate or not. This decoding time is depicted as $L_i$ for the $i$th step.

<figure>
  <img src="https://github.com/osbama/KBM608/blob/main/hands-on/hands-on-7-images/sifl.png?raw=1" alt="Diagram of the steady-state intercircuit feed-forward latency (SIFL) benchmark circuit showing logical state preparation, repeated syndrome extraction rounds, measurement, and feed-forward X gate application with decoder latency intervals labeled L_i" width="1200">
</figure>

Though this circuit is entirely Clifford, meaning the errors could be tracked and applied at the end, it is a good model for the procedure necessary to apply non-Clifford gates like Toffoli gates or $T$ gates. The main bottleneck of a quantum algorithm is the execution of the non-Clifford gates like $T$ gates. Recall from the lesson ["$T$ Gates and Magic State Distillation"](https://github.com/NVIDIA/cuda-q-academic/blob/main/qec101/05_QEC_MSD.ipynb) that $T$ gates are generally prepared by using an asynchronously generated $\ket{T}$ state via a costly process called **magic state distillation (MSD)**. The $\ket{T}$ states are stored and applied as needed via the teleportation circuit shown below.

<figure>
  <img src="https://github.com/osbama/KBM608/blob/main/hands-on/hands-on-7-images/T_teleport.png?raw=1" alt="Circuit diagram of the T gate teleportation protocol showing a magic state input, CNOT gate, measurement, and classically controlled S gate correction" width="600">
</figure>

As demonstrated in the SIFL figure, if $r_{gen}$ < $r_{proc}$, then each step continues with a constant decoder latency $L_i$ (Top). If $r_{gen}$ > $r_{proc}$, a buildup of unprocessed syndromes occurs (bottom) and each subsequent decoder step becomes slower and slower ($L_i$< $L_{i+1}$) before the next $X$ gate is applied.

In Terhal's paper "[Quantum Error Correction for Quantum Memories](https://arxiv.org/pdf/1302.3428)", the argument is made that this backlog becomes a serious problem and grows exponentially with the number of qubits. This means that a decoder with a sufficient throughput is a non-negotiable for QEC.

The final metric "reaction time" essentially sets the limit on the wall clock time for the QPU. It is likely that FTQC circuits will be overwhelmingly limited by application of $T$ or other non-Clifford gates that require feed-forward information from the decoder before proceeding to the next gate.

The reaction time is composed of two main components. First, the latency which is the time from the last syndrome produced by the QPU to the decoder returning its result ($T_{decode}$). The second is the communication overhead from the decoder to the QPU via the classical control system ($T_{latency}$).

$$ T_{reaction}=T_{decode}+T_{latency}$$

Though reaction time is not as critical as throughput, it may be a major practical limitation for completing quantum algorithms with many non-Clifford gates in a reasonable amount of time.

A final honorable mention in the discussion of metrics is scalability. Though, not a metric per se, it is important to remember that any decoder, no matter how promising, needs to scale. Thus, today's research efforts should be directed towards the methods which have the greatest potential to scale and avoid those with proven limitations.


### Exercise 6

Use the `QECSimulator` below to explore the **exponential backlog** problem. The simulator essentially tracks the system load (white line) which consists of all the syndromes currently being decoded and the backlog as a function of wall clock time (of a hypothetical simulated QPU) based on the device parameters you set. The green bars are the time for each decoding round.

To setup the simulation, input the `syndrome_time_us` which is the time in microseconds for a single syndrome extraction from the QPU (this defines $r_{gen}$). The `min_batch_size` specifies the number of syndromes needed to send a batch to the decoder. `decode_func` is a lambda expression which computes how long it will take to decode all $n$ syndromes. This is an important variable to toggle, as the most accurate algorithmic decoders can scale exponentially in terms of the number of syndromes they are decoding. `max_batches_to_run` ensures the simulation stops at a reasonable point. Keep `n_processors` equal to 1 for now.

Run the simulation for the different situations presented below. In the first cell, run the case where the decoder scales linearly with the number of syndromes to decode but is faster than the syndrome generation rate. What happens to the system load and the decoding time?


In [ ]:
# EXERCISE 6

print("Running Scenario 1...")
sim1 = QECSimulator(
    syndrome_time_us=1.0,
    min_batch_size=20,
    decode_func=lambda n: n * 0.8,
    max_batches_to_run=15,
    n_processors=1
)
sim1.run()
sim1.plot_results("Scenario 1: Serial Linear (r_proc > r_gen)")

Now, keep the decode function linear, but change the scaling factor so it is slightly slower than the syndrome generation rate. What happens to the system load and batch decoding times?

In [ ]:
print("Running Scenario 2...")
sim1 = QECSimulator(
    syndrome_time_us=1.0,
    min_batch_size=20,
    decode_func=lambda n: n * 1.2,
    max_batches_to_run=15,
    n_processors=1
)
sim1.run()
sim1.plot_results("Scenario 2: Serial Linear (r_proc < r_gen)")

Now, make the decoder quadratic with the following decode function to better resemble a decoder in practice deployment: $f(n) = 0.01 *n^2$.

In [ ]:
print("Running Scenario 3...")
sim1 = QECSimulator(
    syndrome_time_us=1.0,
    min_batch_size=20,
    decode_func=lambda n: 0.01*n ** 2,
    max_batches_to_run=15,
    n_processors=1
)
sim1.run()
sim1.plot_results("Scenario 3: Serial Quadratic (r_proc > r_gen)")

Even a quadratic scaling decoder can perform well if it is handling a small enough batch of syndromes to keep up, but try changing the prefactor to $f(n) = n^2$ and see what happens.

In [ ]:
print("Running Scenario 4...")
sim1 = QECSimulator(
    syndrome_time_us=1.0,
    min_batch_size=20,
    decode_func=lambda n: n ** 2,
    max_batches_to_run=10,
    n_processors=1
)
sim1.run()
sim1.plot_results("Scenario 4: Serial Quadratic (r_proc << r_gen)")

The simulation can decode the first batch reasonably fast, but the backlog is so severe, the second batch takes 160,000 microseconds! And the simulation continues to blow up after this. Clearly, a more effective method is needed when accurate decoders are required and $r_{proc} < r_{gen}$.

One additional note. The 1 microsecond times for syndrome extraction are in the ballpark for superconducting devices. For slower modalities like ion traps, the same decoder setup may work fine simply because it takes so much more time to perform syndrome extraction. Try running the analysis one more time but using a syndrome extraction time of 200 microseconds.

In [ ]:
print("Running Scenario 5...")
sim1 = QECSimulator(
    syndrome_time_us=200.0,
    min_batch_size=20,
    decode_func=lambda n: n ** 2,
    max_batches_to_run=15,
    n_processors=1
)
sim1.run()
sim1.plot_results("Scenario 5: Serial Quadratic (Ion Trap) (r_proc > r_gen)")

Thus, different decoders may be better suited for different qubit modalities and a whole host of other factors.

## Parallel Window Decoding

The throughput problem is serious, and finding clever ways to increase decoder throughput is necessary to ensure the viability of real-time decoding. Accelerated computing has helped solve simulation problems across all domains of science by massively parallelizing scientific computing and boosting throughput by orders of magnitude. The QEC field is primed to benefit from such an approach as well and this section will build towards a scheme called parallel window decoding which can ameliorate the throughput problem.

Building up from a simple model. If a circuit was entirely composed of Clifford gates, it could be run and all syndromes be decoded after its completion as a single postprocessing step. Unfortunately, the benefits of quantum algorithms comes from non-Clifford gates like $T$ gates which require feedback from a decoder at each application. This means a decoder needs to work alongside the QPU in realtime. The magic states required for application of $T$ gates can also decohere if the decoder backlog is too high. Thus, clever parallel decoding schemes are required to solve the problem.

Let's consider decoding for a single set of 6 stabilizer rounds performed before some $T$ gate. One approach to decoding these could be waiting for all 6 rounds to finish and then decoding them all at once. (The `QECSimulator` above does currently.) This block decoding will be the slowest approach possible as the decoder must process a massive parity (that covers all syndromes) at once.

<figure>
  <img src="https://github.com/osbama/KBM608/blob/main/hands-on/hands-on-7-images/alldecode.png?raw=1" alt="Diagram showing block decoding where all 6 syndrome extraction rounds are collected before decoding begins as a single batch" width="500">
</figure>

Aside from the more difficult decoding task, this approach cannot start until all of the syndromes are collected so there is no chance for a headstart.

A second, far more common approach is called **sliding window decoding**. The idea is to decode slices of the syndrome data and feed the results into the next decoding task. One advantage is that the decoding task is much smaller as the "sliding window" covers fewer syndromes at the same time. A second advantage is that the decoder can start working on the first window as soon as it is ready rather than waiting for all of the syndrome data to be generated.

<figure>
  <img src="https://github.com/osbama/KBM608/blob/main/hands-on/hands-on-7-images/sliding.png?raw=1" alt="Diagram showing sliding window decoding where overlapping windows of syndrome data are decoded sequentially, with each window covering a subset of rounds" width="500">
</figure>

Even though the decoder gets a head start, the decoding must still occur in serial as each decoding step cannot begin until the previous is finished. Thus, sliding window decoding will quickly run into scaling problems.

The solution to this is **parallel window decoding** as presented in the paper entitled ["Parallel window decoding enables scalable fault tolerant quantum computation"](https://www.nature.com/articles/s41467-023-42482-1). Parallel window decoding can be applied with respect to time (temporal) or space (spatial). For this lesson we will focus on temporal parallelism only, but know that similar techniques could be used to break down a large QEC code patch spatially.

In temporal parallel window decoding, the entire block of syndrome history is obtained and then broken down into sub-blocks that can run in parallel on $N_{proc}$ number of processors in two steps: decoding and cleanup.

<figure>
  <img src="https://github.com/osbama/KBM608/blob/main/hands-on/hands-on-7-images/pw_decoding_diagram.png?raw=1" alt="Diagram of parallel window decoding showing syndrome history divided into sub-blocks assigned to multiple processors for parallel decode and cleanup steps" width="500">
</figure>

The decoding step first commits error assignments in specified regions while the second cleans up the boundaries to rectify errors between the commit regions as shown in the image below.

<figure>
  <img src="https://github.com/osbama/KBM608/blob/main/hands-on/hands-on-7-images/pw_schematic.png?raw=1" alt="Schematic of parallel window decoding showing alternating commit regions (green) and buffer regions, with the two-step decode-then-cleanup procedure illustrated across multiple processors" width="1000">
</figure>

Consider the central commit region in green. There are $n_{com}$ syndromes needed for the decode step. The commit regions are then flanked by buffer regions where syndromes are partially processed, but the results cannot be certain yet. The commit regions are the solved boundaries for the syndromes processed in the second cleanup step. Here, $n_w$ syndromes are processed to determine errors in the two overlapping buffer regions plus any additional syndromes between them.

As you can see from the figure, this pattern repeats for as many processors and sub-blocks as necessary. For counting purposes, the smallest sub-block that can be processed in this manner consists of $n_{com} + n_w$ syndromes. There are many potential choices for the size of these regions. For example, the ["Parallel window decoding enables scalable fault tolerant quantum computation"](https://www.nature.com/articles/s41467-023-42482-1) paper sets $n_{com} = d$ and $n_w = 3d$. Techniques like "[temporal encoding of lattice surgery](https://journals.aps.org/prxquantum/abstract/10.1103/PRXQuantum.3.010331)" can allow for a smaller number of syndrome extraction rounds to be used, improving the throughput of the decoder even more by minimizing the sub-block decoding task.

To avoid an exponential backlog if we have a single decoder, the following inequality must hold.

$$ (n_{com} + n_w)t_{\text{syndrome-extraction}} \geq 2 \cdot t_{\text{decode}} $$

At face value this seems like a more challenging decoding task relative to the standard approach as $t_{\text{decode}}$ now has a factor of 2. The trick is that this construction, though requiring two steps, can be attacked with any number of processors $N_{proc}$ so instead the following must hold.

$$ N_{proc}(n_{com} + n_w)t_{\text{syndrome-extraction}} \geq 2 \cdot t_{\text{decode}} $$

This means that a slow decoder using parallel window decoding can still avoid the exponential backlog problem by using an arbitrary number of decoders in parallel up to the limit of the smallest sub-block decoding task. Such an approach has replaced the viability of sliding window and provides a much more promising path to scalable QEC.



### Exercise 7:

Use the QEC Simulator above and determine how many processors are required to keep up with the quadratic scaling decoder you tested earlier. Note that the simulator considers the fact that two decoding rounds are required for parallel window decoding, however the simulator is simplified and divides the syndromes to process into $N$ blocks approximating the procedure to form a buffer zone.



In [ ]:
# EXERCISE 7

print("Running Scenario 6...")
sim1 = QECSimulator(
    syndrome_time_us=1.0,
    min_batch_size=20,
    decode_func=lambda n:  n ** 2,
    max_batches_to_run=15,
    n_processors=7
)
sim1.run()
sim1.plot_results("Scenario 6: Parallel Quadratic (r_proc > r_gen)")

The key takeaway here is that it is often necessary to take a massive decoding task and spend the overhead to distribute it across AI supercomputing resources to avoid the exponential backlog. This is a primary motivation for why GPUs are powerful tools for QEC, even if they might have greater latency than other alternatives.

## Decoder Latency and Reaction Time

Assuming the throughput problem is solved, latency becomes the next critical factor. The latency is the time it takes from the QPU producing the last syndrome to when the decoder obtains a result for this syndrome. This is closely related to a similar quantity called reaction time which also includes the time it takes to transmit the result through the classical control systems.

Rerun your parallel decoding simulation above, but this time use 100 decoders. What happens to the decode time?

In [ ]:
print("Running Scenario 7...")
sim1 = QECSimulator(
    syndrome_time_us=1.0,
    min_batch_size=20,
    decode_func=lambda n:  n ** 2,
    max_batches_to_run=25,
    n_processors=100
)
sim1.run()
sim1.plot_results("Scenario 7: Parallel Quadratic - many processors (r_proc >> r_gen)")

Notice each decoder task is much faster so the result can also be returned to the QPU much faster and progress the quantum computation. Note that more processors can improve the latency, but they are still limited by twice the time it takes to decode a fundamental block. Though this simulator can arbitrarily compute smaller times to give a qualitative depiction of this, in practice, the previous discussion about the size of the buffer and commit regions determines the limit on latency.

A quantum algorithm will be constrained regardless, but there are a number of tradeoffs that can be considered. If the MSD process is slow and few $\ket{T}$ states are on hand, then MSD becomes the bottleneck for QPU computation speed. If the number of $\ket{T}$ states is abundant, then the decoder's reaction time becomes the bottleneck as the decoder must return its result before the next $T$ gate can be applied.

This means that decoding will usually be the primary bottleneck for the QPU wall clock time and might be the difference between an algorithm completing in a reasonable amount of time or not. This is why decoders need to be as fast as possible (even when the throughput problem is solved) and connect to the quantum control devices with interconnects optimized for latency such as [NVIDIA's NVQLink](https://www.nvidia.com/en-us/solutions/quantum-computing/nvqlink/).

Additional space and time tradeoffs can be considered with methods like autocorrected $T$ gates, where $T$ gates proceed without resolving the necessary Clifford correction which is instead tracked as part of the quantum algorithm, speeding up the wall clock time. However, this time savings requires a space cost in the form of additional ancillas for every $T$ gate. Neither approach resolves the underlying constraints but, like many aspects of quantum computing, allows for selection of tradeoffs depending on the application.

## Conclusion

After completing this lab, you should now have a better understanding for why decoding is so difficult and what specific metrics need to be considered when using a decoder in practice. A key takeaway is that the throughput problem is an absolute dealbreaker for decoders. If a decoder cannot process syndromes as fast as they are obtained from the QPU, it will result in an exponential backlog that grinds everything to a halt.

Parallel window decoding is an innovative solution to this problem that allows many processors working in tandem to solve the throughput problem.

Response time and accuracy are also very important and might impose severe practical limitations on a QEC workflow, but these are secondary to throughput.